# Hadamard edge detection

Encode the full 128 × 128 photo and sample the cyclic edge detector.

Set up local FlagQuantum execution and image plotting.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.image_processing.quantum_hadamard_edge_detection.overview import workflow
from examples.synthesis.applications.image_processing.quantum_hadamard_edge_detection.overview.main import print_report, save_figures
from examples.synthesis.applications.image_processing.quantum_hadamard_edge_detection.overview.quantum import AXIS_BITS

Load the original RGB asset.

In [ ]:
photo = workflow.load_photo()
print('Photo:', tuple(photo.shape))

Reproduce the original posterization preview. Keep the source photo for amplitude encoding.

In [ ]:
preview = workflow.threshold_preview(photo)
assert (preview[0] == photo[0]).all()

Convert unmodified pixel magnitudes to unit amplitudes.

In [ ]:
pixels = workflow.pixel_amplitudes(photo)
print('Amplitude norm:', float(pixels.norm()))

Compute the original seven bits along each axis.

In [ ]:
print('Pixel qubits along each axis:', AXIS_BITS)

Preserve all 16,384 pixel amplitudes.

In [ ]:
print('Pixel count:', pixels.numel())
assert pixels.numel() == 16_384

Prepare the real amplitude vector and compose both cyclic shifts and their edge flag.

In [ ]:
result = workflow.run()
print('Complete state error:', result.state_error)

Check the three restored compiler work bits before taking the direction-bit marginal.

In [ ]:
print('Work probability:', result.work_probability)
assert result.work_probability < 1e-24

Apply the source width bound to the complete circuit.

In [ ]:
assert result.resources.n_qubits <= 20

Report the actual prepared and composed resources.

In [ ]:
print_report(result)

Inspect the emitted rotation and controlled shift resources.

In [ ]:
print('Preparation operations:', dict(result.preparation_resources.operation_counts))
print('Full circuit operations:', dict(result.resources.operation_counts))

Retain all 200,000 finite local draws and report actual edge-flag events.

In [ ]:
print('Total draws:', workflow.SHOTS)
print('Observed edge events:', result.observed_edge_shots)

Reconstruct the sampled image from square roots of measured joint frequencies.

In [ ]:
print('Raw nonzero edge pixels:', int((result.sampled_edge > 0).sum()))

Plot the sampled pixel magnitude histogram.

In [ ]:
figures = Path('notebook-figures')
save_figures(result, figures)
display(Image(filename=str(figures / 'edge-histograms.png')))

Threshold the joint magnitudes at 0.003 and inspect the thresholded histogram.

In [ ]:
print('Edge threshold:', workflow.THRESHOLD)
assert bool(((result.thresholded_edge == 0) | (result.thresholded_edge > workflow.THRESHOLD)).all())

Display the transposed finite-shot result with the exact edge comparison.

In [ ]:
display(Image(filename=str(figures / 'edge-images.png')))

Check that the original photo stayed unmodified through the full workflow.

In [ ]:
assert bool((result.photo == photo).all())
print('Original photo retained:', tuple(result.photo.shape))